# PCA## AimPrincipal Component Analysis for dimensionality reduction.

## Problem TypeDimensionality Reduction

## Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

sns.set_style("whitegrid")


## Dataset Input Options

In [ ]:
# Cell A: Load toy dataset (Wine dataset)
# Keep this as default for quick exam execution.
raw = load_wine(as_frame=True)
if hasattr(raw, "frame") and raw.frame is not None:
    df = raw.frame.copy()
else:
    df = pd.DataFrame(raw.data, columns=raw.feature_names)
    df["target"] = raw.target
TARGET_FOR_COLOR = "target" if "target" in df.columns else None
DATA_SOURCE = "toy"
print("Using toy dataset. Shape:", df.shape)


In [ ]:
# Cell B: Load local CSV dataset
# Change USE_CSV to True only when you have an exam CSV file.
USE_CSV = False
CSV_PATH = "your_dataset.csv"
TARGET_FOR_COLOR = None

if USE_CSV:
    df = pd.read_csv(CSV_PATH)
    DATA_SOURCE = "csv"
    print("Using CSV dataset. Shape:", df.shape)
else:
    print("CSV mode is OFF. Continuing with toy dataset.")


## Dataset Overview / One-cell EDA

In [ ]:
print("Shape:", df.shape)
print("\nHead:")
display(df.head())
print("\nInfo:")
df.info()
print("\nDescribe:")
display(df.describe(include="all"))
print("\nMissing values:")
print(df.isnull().sum())


## Preprocessing (Generic and Reusable)

In [ ]:
if TARGET_FOR_COLOR is not None and TARGET_FOR_COLOR in df.columns:
    X = df.drop(columns=[TARGET_FOR_COLOR])
    y_color = df[TARGET_FOR_COLOR]
else:
    X = df.copy()
    y_color = None

numeric_cols = X.select_dtypes(include=["number"]).columns
categorical_cols = X.select_dtypes(exclude=["number"]).columns

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols),
    ]
)

X_processed = preprocessor.fit_transform(X)
if hasattr(X_processed, "toarray"):
    X_processed = X_processed.toarray()

print("Processed feature shape:", X_processed.shape)


## Apply PCA

In [ ]:
n_components = 2
pca = PCA(n_components=n_components, random_state=42)
X_reduced = pca.fit_transform(X_processed)
print("PCA completed.")


## PCA Results

In [ ]:
reduced_df = pd.DataFrame(X_reduced, columns=["PC1", "PC2"])
display(reduced_df.head())


## Explained Variance

In [ ]:
evr = pca.explained_variance_ratio_
print("Explained Variance Ratio:", evr)
print("Total Explained Variance:", evr.sum())


## Visualization

In [ ]:
plot_df = reduced_df.copy()
if y_color is not None:
    plot_df["Label"] = y_color.values
plt.figure(figsize=(7, 5))
if "Label" in plot_df.columns:
    sns.scatterplot(data=plot_df, x="PC1", y="PC2", hue="Label", palette="tab10")
else:
    sns.scatterplot(data=plot_df, x="PC1", y="PC2")
plt.title("PCA 2D Projection")
plt.tight_layout()
plt.show()


## InterpretationPCA successfully reduced dimensionality while preserving most of the variance.

## SuggestionsTry different numbers of components, analyze scree plots, and use for visualization or preprocessing.

## ConclusionThis notebook provides a reusable dimensionality reduction workflow for toy and CSV datasets in exam settings.